# F0 · Frozen DINOv2-S/14 + P3 + E4 + horizontal-flip TTA
Self-contained Colab notebook. Default: train a fresh F0 classifier, then compare original inference with equal-probability original/flip TTA and export both CSVs. A fresh run is **not guaranteed to reproduce the historical98% validation or0.94693Kaggle scores**. To evaluate the exact previous best model instead, select `MODE='load_checkpoint'` and provide its checkpoint. No automated Kaggle submission, external training data, pseudo-labels or validation/test training. Existing experiments remain unchanged.

## 0. Environment
Choose Runtime → Change runtime type → T4 GPU. Keep Colab’s installed compatible torch/torchvision pair.

In [ ]:
%pip -q install scikit-learn pandas pillow tqdm matplotlib kagglehub


In [ ]:
import os, platform, torchvision
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from datetime import datetime, timezone
import uuid
import random, hashlib, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
from IPython.display import display



## 1. Paths and execution mode
Default downloads the course dataset using Colab Secret `KAGGLE_API_TOKEN`; enable notebook access and accept the competition rules. Set `DOWNLOAD_FROM_KAGGLE=False` for an extracted dataset and edit DATA_ROOT. No credential values belong in code. Optional Drive persists features/checkpoints. `MODE='train_f0'` needs no old checkpoint. `MODE='load_checkpoint'` skips all feature extraction/training and checks the expected original score before test prediction.

In [ ]:
MODE = 'train_f0'  # Or 'load_checkpoint' to use the exact previous F0 model.
USE_DRIVE = False
DOWNLOAD_FROM_KAGGLE = True
DATA_ROOT = Path('/content/car_data')
STORAGE_ROOT = Path('/content')
F0_CHECKPOINT = Path('/content/dinov2_p3_e4_aug123_head2026.pt')  # Used only in load mode.
EXPECTED_LOADED_ACCURACY = 0.98  # Actual verified starting score for the supplied checkpoint.
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/Car-Classification-Benchmark')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
        except Exception as exc:
            raise RuntimeError('Enable Colab Secret KAGGLE_API_TOKEN, or disable download and set DATA_ROOT to the extracted course data.') from exc
    DATA_ROOT = Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))
assert MODE in ('train_f0','load_checkpoint')
if MODE == 'load_checkpoint' and not F0_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Upload the original checkpoint at {F0_CHECKPOINT}, or explicitly choose MODE=train_f0 for a new training run.')
for split in ('train','val','test'):
    if not (DATA_ROOT/split).is_dir(): raise FileNotFoundError(f'Missing {DATA_ROOT/split}; set DATA_ROOT or enable the download.')
assert (DATA_ROOT/'sample_submission.csv').is_file()
SEED_AUG,SEED_HEAD = 123,2026
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16
EXTRACTION_BATCH_SIZE = BATCH_SIZE
EXPERIMENT = 'P3_geometry_multiview'
EPOCHS,LEARNING_RATE,WEIGHT_DECAY,PATIENCE = 35,1e-3,1e-3,7
MODEL_NAME = 'dinov2_vits14'
FEATURE_DIM = 384
HUB_REF = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
HUB_REPO = f'facebookresearch/dinov2:{HUB_REF}'
WEIGHTS_SOURCE = 'https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth'
N_AUG,INCLUDE_ORIGINAL = 4,True
PIPELINE_VERSION = 'F0-TTA-v1'
FORCE_REEXTRACT = False
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
OUTPUT_DIR = STORAGE_ROOT/'student_outputs_F0_TTA'/RUN_ID
CACHE_DIR = STORAGE_ROOT/'student_outputs_F0_TTA'/'cache'
OUTPUT_DIR.mkdir(parents=True,exist_ok=False);CACHE_DIR.mkdir(parents=True,exist_ok=True)
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
set_seed(SEED_AUG)
print('MODE:',MODE,'Data:',DATA_ROOT,'Output:',OUTPUT_DIR,'Device:',DEVICE)
print('Python:',platform.python_version(),'PyTorch:',torch.__version__)


## 2. Dataset and P3 preprocessing
20prefixed classes,700training/200validation images. P3 uses one deterministic view plus four random geometric views; no ColorJitter. Validation/test always use bicubic Resize256→CenterCrop224 and ImageNet normalization.

In [ ]:
# preprocessing change
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=val_transform)
preprocess = val_transform

assert train_ds.class_to_idx == val_ds.class_to_idx, 'train/ and val/ class folders must match'
NUM_CLASSES = len(train_ds.classes)
print('Classes:', NUM_CLASSES, '| Train:', len(train_ds), '| Val:', len(val_ds))
print('First class folders:', train_ds.classes[:5])
assert NUM_CLASSES == 20, f'Expected 20 classes, got {NUM_CLASSES}'

# Class-folder names from the instructor's export are prefixed by 000_, 001_, ...
# These prefixes make ImageFolder indices equal the Kaggle integer labels.
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch at {folder}; ask your instructor'

assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected original 700/200 split.'

## 3. Original sample-submission ordering
Test labels are never used. The original CSV is required; string IDs, uniqueness and file coverage are checked.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
test_dir = DATA_ROOT / 'test'
test_files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
assert test_files, f'No images found inside {test_dir.resolve()}'

sample_path = DATA_ROOT / 'sample_submission.csv'
assert sample_path.exists(), 'Original sample_submission.csv is required.'
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype={'image_id': str})
    assert {'image_id','label'}.issubset(sample.columns)
    required_ids = sample['image_id'].astype(str).tolist()
else:
    required_ids = sorted(p.stem for p in test_files)

by_id = {p.stem: p for p in test_files}
assert len(by_id) == len(test_files), 'Two test filenames share the same image_id stem'
assert len(set(required_ids)) == len(required_ids), 'Duplicate image_id in sample_submission.csv'
assert set(required_ids) == set(by_id), ('Test image IDs do not match sample_submission.csv',
                                         len(set(required_ids)-set(by_id)), len(set(by_id)-set(required_ids)))
print('Unlabeled test images:', len(required_ids))
print('Submission ID example:', required_ids[:3])

class TestImages(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        p = by_id[self.ids[index]]
        with Image.open(p) as im: image = preprocess(im.convert('RGB'))
        return image

test_ds = TestImages(required_ids)
assert len(required_ids)==len(sample)


## 4. Official pretrained frozen DINOv2-S/14
Use the same permitted SSL architecture as F0,384-D CLS features, pinned official upstream source, no random-weight fallback. Hub caches downloads. No fine-tuning.

In [ ]:
backbone = torch.hub.load(HUB_REPO,MODEL_NAME,pretrained=True,trust_repo=True,skip_validation=True).to(DEVICE)
backbone.requires_grad_(False).eval()
assert backbone.embed_dim==FEATURE_DIM
with torch.inference_mode(): probe=backbone(torch.zeros(1,3,224,224,device=DEVICE))
assert probe.shape==(1,384)
del probe
def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_SHA256 = weight_digest(backbone)
print('Frozen backbone:',MODEL_NAME,'Feature dim:',FEATURE_DIM)


## 5. Safe P3 feature cache · training mode only
Cache identity includes actual pretrained weights/source, transforms, augmentation seed, view counts/layout, class mapping and content-hashed manifests. Invalid caches regenerate. Head-only changes reuse features. Source/metadata checks prevent stale in-memory features. No validation/test statistics enter fitting.

In [ ]:
if MODE == 'train_f0':
    def file_sha256(path):
        h = hashlib.sha256()
        with open(path, 'rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''): h.update(chunk)
        return h.hexdigest()

    def feature_metadata():
        ordered = ([Path(p) for p, _ in train_ds.samples],
                   [Path(p) for p, _ in val_ds.samples], [by_id[i] for i in required_ids])
        manifests = [[{'path': str(p.relative_to(DATA_ROOT)), 'sha256': file_sha256(p)}
                      for p in paths] for paths in ordered]
        sets = [{entry['sha256'] for entry in split} for split in manifests]
        assert not any(sets[i] & sets[j] for i,j in ((0,1),(0,2),(1,2))), 'Duplicate image content across splits'
        return {'pipeline': PIPELINE_VERSION, 'backbone': MODEL_NAME, 'feature_dim': FEATURE_DIM,
                'hub_repo': HUB_REPO, 'weights_source': WEIGHTS_SOURCE, 'weights_sha256': WEIGHTS_SHA256,
                'preprocessing': 'P3_geometry_multiview', 'train_transform': repr(train_transform),
                'val_transform': repr(val_transform), 'seed_aug': SEED_AUG, 'n_aug': N_AUG,
                'include_original': INCLUDE_ORIGINAL, 'layout': 'view-major',
                'class_to_idx': dict(train_ds.class_to_idx), 'manifest': manifests,
                'test_ids': list(required_ids), 'batch_size': EXTRACTION_BATCH_SIZE,
                'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
                'pillow': Image.__version__, 'device': DEVICE}

    def metadata_digest(meta):
        return hashlib.sha256(json.dumps(meta, sort_keys=True).encode()).hexdigest()

    @torch.inference_mode()
    def extract(loader, desc):
        backbone.eval()
        assert not any(p.requires_grad for p in backbone.parameters())
        embeddings = []
        for batch in tqdm(loader, desc=desc):
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            z = backbone(x.to(DEVICE)).float().cpu()
            assert z.ndim == 2 and z.shape[1] == FEATURE_DIM and torch.isfinite(z).all()
            embeddings.append(z)
        return torch.cat(embeddings)

    def valid_cache(payload, meta):
        try:
            assert payload['metadata'] == meta
            views = N_AUG + int(INCLUDE_ORIGINAL)
            for key,n in [('X_train',len(train_ds)*views), ('X_val',len(val_ds)), ('X_test',len(required_ids))]:
                x = payload[key]
                assert x.dtype == torch.float32 and x.shape == (n,FEATURE_DIM) and torch.isfinite(x).all()
            assert torch.equal(payload['y_train'], torch.tensor(train_ds.targets).repeat(views))
            assert torch.equal(payload['y_val'], torch.tensor(val_ds.targets))
            return True
        except (KeyError, AssertionError, TypeError, AttributeError):
            return False

    def get_features(force=False):
        meta = feature_metadata()
        signature = metadata_digest(meta)
        path = CACHE_DIR / f'F0_dinov2_vits14_p3_aug{SEED_AUG}_{signature}.pt'
        if path.exists() and not force:
            try:
                payload = torch.load(path, map_location='cpu', weights_only=True)
                if valid_cache(payload, meta):
                    print('Loaded compatible cache:', path)
                    return payload, signature, path
                print('Incompatible cache; regenerating')
            except Exception as exc:
                print('Unreadable cache; regenerating:', type(exc).__name__)
        set_seed(SEED_AUG)
        kwargs = dict(batch_size=EXTRACTION_BATCH_SIZE, shuffle=False, num_workers=0,
                      pin_memory=(DEVICE == 'cuda'))
        views = []
        original_transform = train_ds.transform
        try:
            if INCLUDE_ORIGINAL:
                train_ds.transform = val_transform
                views.append(extract(DataLoader(train_ds, **kwargs), 'Original training view'))
            train_ds.transform = train_transform
            for index in range(N_AUG):
                views.append(extract(DataLoader(train_ds, **kwargs), f'Augmented training view {index+1}'))
        finally:
            train_ds.transform = original_transform
        payload = {'metadata': meta, 'X_train': torch.cat(views),
                   'X_val': extract(DataLoader(val_ds, **kwargs), 'Validation'),
                   'X_test': extract(DataLoader(test_ds, **kwargs), 'Test'),
                   'y_train': torch.tensor(train_ds.targets).repeat(N_AUG + int(INCLUDE_ORIGINAL)),
                   'y_val': torch.tensor(val_ds.targets)}
        assert valid_cache(payload, meta)
        temp = path.with_suffix('.' + uuid.uuid4().hex + '.tmp')
        torch.save(payload, temp); temp.replace(path)
        print('Freshly extracted:', path)
        return payload, signature, path

    features, FEATURES_SIGNATURE, CACHE_PATH = get_features(FORCE_REEXTRACT)
    X_train, X_val, X_test = [features[k] for k in ('X_train','X_val','X_test')]
    y_train, y_val = features['y_train'], features['y_val']
    assert X_train.shape == (3500,384) and y_train.shape == (3500,)
    assert X_val.shape == (200,384) and X_test.shape == (len(sample),384)
    print(X_train.shape, X_val.shape, X_test.shape)
    FEATURES_MODEL_NAME = MODEL_NAME
    dataset_signature = lambda: metadata_digest(feature_metadata())
else:
    print('Checkpoint mode: feature extraction and classifier training will be skipped.')

## 6. Train-only standardization and E4 head
Head: Linear384→128, BatchNorm, ReLU, Dropout.3, Linear128→20. Default training uses AdamW lr/decay.001, batch64,35epochs, patience7; accuracy-only checkpoint selection matches F0. Fresh head/optimizer with head seed2026.

In [ ]:
HEAD_CONFIGS={'E4_mlp_bn_drop03':{'hidden_dim':128,'use_bn':True,'dropout':0.3}}
def build_head(experiment,input_dim,num_classes):
    assert experiment=='E4_mlp_bn_drop03' and input_dim==384 and num_classes==20
    return nn.Sequential(nn.Linear(input_dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,num_classes))
if MODE=='train_f0':
    feature_mean = X_train.mean(dim=0)
    feature_std = X_train.std(dim=0, unbiased=False).clamp_min(1e-6)
    Z_train = (X_train - feature_mean) / feature_std
    Z_val = (X_val - feature_mean) / feature_std
    Z_test = (X_test - feature_mean) / feature_std
    print('Features standardized without validation/test leakage.')

## 7. Train a new F0 classifier or restore the original
No previous classifier is overwritten. Loaded mode validates the original model family, E4 shape, seeds, class mapping and384-D normalization. If checkpoint metadata differs, diagnose rather than relabel it as F0.

In [ ]:
if MODE=='train_f0':
    from torch.utils.data import TensorDataset
    training_loader = DataLoader(TensorDataset(Z_train, y_train), batch_size=64,
                                 shuffle=True, num_workers=0)
    HEAD_EXPERIMENT = "E4_mlp_bn_drop03"

    # Ensure features match the selected backbone
    assert FEATURES_MODEL_NAME == MODEL_NAME
    assert FEATURES_SIGNATURE == dataset_signature(), (
        "Feature configuration changed. "
        "Rerun Sections 5 and 6."
    )

    # Reset classifier randomness
    random.seed(SEED_HEAD)
    np.random.seed(SEED_HEAD)
    torch.manual_seed(SEED_HEAD)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED_HEAD)

    # Reproducible training data order
    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED_HEAD)

    training_loader = DataLoader(
        TensorDataset(Z_train, y_train),
        batch_size=64,
        shuffle=True,
        num_workers=0,
        generator=loader_generator
    )

    # Fresh classifier for each experiment
    head = build_head(
        HEAD_EXPERIMENT,
        FEATURE_DIM,
        NUM_CLASSES
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    print("Experiment:", HEAD_EXPERIMENT)
    print("Head seed:", SEED_HEAD)
    print("Architecture:")
    print(head)

    best_accuracy = -1.0
    best_weights = None
    bad_epochs = 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        head.train()
        running_loss = 0.0
        for xb, yb in training_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(yb)
        head.eval()
        with torch.inference_mode():
            val_pred = head(Z_val.to(DEVICE)).argmax(dim=1).cpu().numpy()
        val_accuracy = accuracy_score(y_val.numpy(), val_pred)
        history.append({'epoch': epoch, 'train_loss': running_loss / len(Z_train),
                        'val_accuracy': val_accuracy})
        print(f'Epoch {epoch:02d}/{EPOCHS} | train loss: {history[-1]["train_loss"]:.4f} | val accuracy: {val_accuracy:.2%}')
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_weights = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print('Early stopping: validation accuracy did not improve')
                break

    head.load_state_dict(best_weights)
    head.eval()
    print(f'Best validation accuracy: {best_accuracy:.2%}')
    # pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

    # torch.save({'head_state_dict': best_weights, 'feature_mean': feature_mean,
    #             'feature_std': feature_std, 'class_to_idx': train_ds.class_to_idx,
    #             'backbone': MODEL_NAME}, OUTPUT_DIR / 'linear_head.pt')

    RUN_NAME = (
        f"F0_dinov2_vits14_p3_e4_aug{SEED_AUG}_head{SEED_HEAD}"
    )

    checkpoint_path = OUTPUT_DIR / f"{RUN_NAME}.pt"
    submission_path = OUTPUT_DIR / f"{RUN_NAME}.csv"

    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / f"{RUN_NAME}_history.csv",
        index=False
    )

    torch.save({
        "head_state_dict": best_weights,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "class_to_idx": train_ds.class_to_idx,
        "backbone": MODEL_NAME,
        "preprocessing": EXPERIMENT,
        "head_experiment": HEAD_EXPERIMENT,
        "head_config": HEAD_CONFIGS[HEAD_EXPERIMENT],
        "seed_aug": SEED_AUG,
        "seed_head": SEED_HEAD,
        "best_accuracy": best_accuracy
    }, OUTPUT_DIR / f"{RUN_NAME}.pt")

    print("Saved experiment:", RUN_NAME)
    expected_original_accuracy=best_accuracy
    checkpoint_path=OUTPUT_DIR/f'{RUN_NAME}.pt'
else:
    saved=torch.load(F0_CHECKPOINT,map_location='cpu',weights_only=True)
    required={'head_state_dict','feature_mean','feature_std','class_to_idx','backbone','seed_aug','seed_head','head_config'}
    assert required<=saved.keys(), 'Incomplete original F0 checkpoint'
    assert saved['backbone']==MODEL_NAME
    assert saved['seed_aug']==123 and saved['seed_head']==2026
    assert saved['head_config']==HEAD_CONFIGS['E4_mlp_bn_drop03']
    assert saved['class_to_idx']==train_ds.class_to_idx
    feature_mean,feature_std=saved['feature_mean'],saved['feature_std']
    assert feature_mean.shape==feature_std.shape==(384,)
    assert torch.isfinite(feature_mean).all() and torch.isfinite(feature_std).all() and (feature_std>0).all()
    head=build_head('E4_mlp_bn_drop03',384,20).to(DEVICE)
    head.load_state_dict(saved['head_state_dict'],strict=True);head.eval()
    checkpoint_path=F0_CHECKPOINT
    expected_original_accuracy=EXPECTED_LOADED_ACCURACY

## 8. Fixed two-view probability averaging
One original center crop and its horizontal flip, equal weights, no tuning. Head/backbone eval mode keeps Dropout off and BatchNorm unchanged. The standalone functions below use the exact model and saved training statistics. Labels are used only for evaluation, never inference.

In [ ]:
"""Paste/run inside the existing verified F0 notebook; no repo imports required."""
from pathlib import Path
from types import SimpleNamespace
import uuid, math, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
common=SimpleNamespace(accuracy_score=accuracy_score,f1_score=f1_score)

@torch.inference_mode()
def predict_flip_tta(backbone,head,mean,std,loader,device):
    """Loader must contain deterministic preprocessed images, in required order.

    Returns original probabilities and equal original/horizontal-flip average.
    Works with an already verified F0/I0 backbone and its corresponding head/stats.
    Never passes labels to the model or updates normalization statistics.
    """
    backbone.eval();head.eval()
    assert not any(p.requires_grad for p in backbone.parameters()), 'Use a frozen backbone'
    mean,std=mean.to(device),std.to(device)
    assert mean.ndim==std.ndim==1 and mean.shape==std.shape
    assert torch.isfinite(mean).all() and torch.isfinite(std).all() and (std>0).all()
    original,averaged=[],[]
    for index,batch in enumerate(loader):
        images=batch[0] if isinstance(batch,(list,tuple)) else batch
        images=images.to(device)
        p=[]
        for view in (images,images.flip(-1)):
            features=backbone(view).float()
            assert features.ndim==2 and features.shape[1]==len(mean)
            probabilities=head((features-mean)/std).softmax(1)
            assert probabilities.shape==(len(images),20) and torch.isfinite(probabilities).all()
            p.append(probabilities.cpu())
        original.append(p[0]);averaged.append((p[0]+p[1])*0.5)
        if index%40==0: print('TTA_BATCH',index+1,'/',len(loader),flush=True)
    return torch.cat(original),torch.cat(averaged)

def comparison_metrics(target,original,tta):
    target=np.asarray(target)
    first=original.argmax(1).numpy();second=tta.argmax(1).numpy()
    before=first==target;after=second==target
    corrected=int((~before&after).sum());regressed=int((before&~after).sum())
    n=corrected+regressed
    # Exact paired binomial/McNemar test; small-disagreement samples remain uncertain.
    p_value=min(1.,2*sum(math.comb(n,k) for k in range(min(corrected,regressed)+1))/2**n) if n else 1.
    def score(pred):
        return {'accuracy':float(common.accuracy_score(target,pred)),
                'macro_f1':float(common.f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
                'correct':int((pred==target).sum()),'count':len(target)}
    return {'original':score(first),'horizontal_flip_tta':score(second),'corrected':corrected,
            'regressed':regressed,'changed_predictions':int((first!=second).sum()),'exact_paired_p_value':p_value}

def save_submission(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids))
    assert ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')

def run_verified_f0_flip_tta(backbone,head,feature_mean,feature_std,val_ds,test_ds,
                             required_ids,output_dir,device,expected_accuracy=0.98,batch_size=16):
    # Use the exact backbone, head and statistics that produced the verified F0 result.
    assert len(val_ds)==200 and len(test_ds)==len(required_ids)
    output=Path(output_dir)/('F0_flip_TTA_'+uuid.uuid4().hex[:10]);output.mkdir(parents=True,exist_ok=False)
    val_loader=DataLoader(val_ds,batch_size=batch_size,shuffle=False,num_workers=0)
    original,tta=predict_flip_tta(backbone,head,feature_mean,feature_std,val_loader,device)
    measured=comparison_metrics(val_ds.targets,original,tta)
    assert abs(measured['original']['accuracy']-expected_accuracy)<1e-12, 'Starting F0 accuracy was not reproduced; diagnose before predicting test'
    print('Validation:',measured)
    test_loader=DataLoader(test_ds,batch_size=batch_size,shuffle=False,num_workers=0)
    original_test,tta_test=predict_flip_tta(backbone,head,feature_mean,feature_std,test_loader,device)
    save_submission(output/'F0_original_comparison.csv',required_ids,original_test)
    save_submission(output/'F0_flip_TTA_submission.csv',required_ids,tta_test)
    (output/'summary.json').write_text(json.dumps({'validation':measured,'submitted':False,
        'combination':'equal mean of original/flip probabilities','source':'existing verified F0 notebook objects',
        'test_prediction_count':len(required_ids),'test_changed_predictions':int((original_test.argmax(1)!=tta_test.argmax(1)).sum())},indent=2))
    print('Candidate files (not submitted):',output)
    return output


## 9. Paired validation and two candidate Kaggle CSVs
Original inference must reproduce the checkpoint’s expected score (measured during fresh training, or98% by default for the supplied original checkpoint). A failure stops test prediction. TTA is exported even if validation falls slightly, as an explicitly unproven candidate. No upload to Kaggle occurs.

In [ ]:
tta_output=run_verified_f0_flip_tta(backbone,head,feature_mean,feature_std,
    val_ds,test_ds,required_ids,OUTPUT_DIR,DEVICE,
    expected_accuracy=expected_original_accuracy,batch_size=BATCH_SIZE)
summary=json.loads((tta_output/'summary.json').read_text())
summary.update({'mode':MODE,'checkpoint':str(checkpoint_path),'checkpoint_sha256':None,
    'backbone':MODEL_NAME,'hub_ref':HUB_REF,'weights_sha256':WEIGHTS_SHA256,
    'seed_aug':SEED_AUG,'seed_head':SEED_HEAD,'torch':str(torch.__version__),
    'historical_F0_accuracy':0.98,'historical_F0_Kaggle':0.94693,
    'historical_scores_status':'User-reported reference, not guaranteed by a fresh training run'})
h=hashlib.sha256()
with checkpoint_path.open('rb') as stream:
    for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
summary['checkpoint_sha256']=h.hexdigest()
(tta_output/'summary.json').write_text(json.dumps(summary,indent=2))
comparison=pd.DataFrame([{'inference':key,**value} for key,value in summary['validation'].items()
    if key in ('original','horizontal_flip_tta')])
display(comparison)
comparison.to_csv(tta_output/'validation_comparison.csv',index=False)
print('Original candidate:',tta_output/'F0_original_comparison.csv')
print('TTA candidate:',tta_output/'F0_flip_TTA_submission.csv')
print('Not submitted to Kaggle. Keep the source checkpoint and entire output folder.')
assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
assert weight_digest(backbone)==WEIGHTS_SHA256


## 10. Interpretation
On200validation images one correct prediction is0.5percentage points. A small TTA loss does not establish a Kaggle loss; it also does not imply a gain. Compare original and TTA from the **same checkpoint** manually within submission limits. Do not repeatedly tune weights/crops against the leaderboard. Preserve both files and the measured summary in Drive/downloads before Colab ends. There are no bundled real training/TTA scores in this notebook.